In [1]:
from pathlib import Path
import pandas as pd
import re

PROJECT_ROOT = Path("..")

FINAL_KGL_FILE = (
    PROJECT_ROOT
    / "data"
    / "purification"
    / "self_assertion_kgl_FINAL.csv"
)

DEV_SENTENCE_FILE = (
    PROJECT_ROOT
    / "data"
    / "preprocessed"
    / "development_sentences_clean.csv"
)

final_kgl = pd.read_csv(FINAL_KGL_FILE)
dev = pd.read_csv(DEV_SENTENCE_FILE)

print("Final KGL terms:", len(final_kgl))
print("Development sentences:", len(dev))

print("\nKGL columns:")
print(final_kgl.columns.tolist())

print("\nDevelopment columns:")
print(dev.columns.tolist())

Final KGL terms: 62
Development sentences: 157346

KGL columns:
['term', 'is_seed', 'sources', 'weight', 'token_count', 'internal_correlation_recomputed']

Development columns:
['corpus', 'iso3', 'session', 'year', 'filename', 'sentence_id', 'sentence_text', 'sentence_text_lower', 'procedural_intro']


In [2]:
TOKEN_PATTERN = re.compile(r"[a-z]+(?:-[a-z]+)*")

def tokenize_for_kgl(text):
    return TOKEN_PATTERN.findall(str(text).lower())

In [3]:
self_assertion_terms = set(
    final_kgl["term"].str.lower()
)

print("Unique Self-Assertion terms:", len(self_assertion_terms))

Unique Self-Assertion terms: 62


In [4]:
def get_kgl_hits(text):
    tokens = tokenize_for_kgl(text)

    hits = sorted(
        set(tokens)
        & self_assertion_terms
    )

    return hits


dev["sa_hit_terms"] = (
    dev["sentence_text_lower"]
    .apply(get_kgl_hits)
)

dev["sa_hit"] = (
    dev["sa_hit_terms"]
    .apply(lambda x: len(x) > 0)
)

dev["sa_n_unique_hit_terms"] = (
    dev["sa_hit_terms"]
    .apply(len)
)

In [5]:
print("Development sentences:", len(dev))

print(
    "Sentences with ≥1 Self-Assertion KGL hit:",
    dev["sa_hit"].sum()
)

print(
    "Proportion of development sentences retrieved:",
    dev["sa_hit"].mean()
)

Development sentences: 157346
Sentences with ≥1 Self-Assertion KGL hit: 12181
Proportion of development sentences retrieved: 0.07741537757553417


In [6]:
weight_map = (
    final_kgl
    .set_index("term")["weight"]
    .to_dict()
)

def get_weight_sum(hit_terms):
    return sum(
        weight_map[term]
        for term in hit_terms
    )

dev["sa_weight_sum"] = (
    dev["sa_hit_terms"]
    .apply(get_weight_sum)
)

In [7]:
dev = (
    dev
    .sort_values(
        ["filename", "sentence_id"]
    )
    .reset_index(drop=True)
)

In [8]:
dev["previous_sentence"] = (
    dev.groupby("filename")[
        "sentence_text"
    ]
    .shift(1)
)

dev["next_sentence"] = (
    dev.groupby("filename")[
        "sentence_text"
    ]
    .shift(-1)
)

In [9]:
def combine_context(row):

    parts = []

    if pd.notna(row["previous_sentence"]):
        parts.append(
            str(row["previous_sentence"])
        )

    parts.append(
        str(row["sentence_text"])
    )

    if pd.notna(row["next_sentence"]):
        parts.append(
            str(row["next_sentence"])
        )

    return " ".join(parts)


dev["context_pm1"] = dev.apply(
    combine_context,
    axis=1
)

In [10]:
sa_contexts = (
    dev.loc[
        dev["sa_hit"]
    ]
    .copy()
    .reset_index(drop=True)
)

In [11]:
sa_contexts["sa_hit_terms"] = (
    sa_contexts["sa_hit_terms"]
    .apply(lambda x: "; ".join(x))
)

In [12]:
print(
    "Retrieved candidate Self-Assertion contexts:",
    len(sa_contexts)
)

display(
    sa_contexts[
        [
            "iso3",
            "year",
            "filename",
            "sentence_id",
            "sa_hit_terms",
            "sa_weight_sum",
            "sentence_text",
            "context_pm1"
        ]
    ]
    .head(10)
)

Retrieved candidate Self-Assertion contexts: 12181


,iso3,year,filename,sentence_id,sa_hit_terms,sa_weight_sum,sentence_text,context_pm1
0,AFG,2016,AFG_71_2016.txt,38,call,0.658118,We call on the international community to focu...,"As a result, hundreds of extremist militants o..."
1,AFG,2016,AFG_71_2016.txt,39,urge,1.000000,We urge States to honestly implement their int...,We call on the international community to focu...
2,AFG,2016,AFG_71_2016.txt,44,defend,1.000000,Afghanistan has always desired peaceful relati...,We have repeatedly asked our neighboring count...
3,AFG,2016,AFG_71_2016.txt,45,renounce,0.733682,We have also kept the doors of peace and negot...,Afghanistan has always desired peaceful relati...
4,AFG,2016,AFG_71_2016.txt,57,declare,1.000000,We expect the prominent Islamic scholars who w...,Islam also considers the killing of one innoce...
5,AFG,2016,AFG_71_2016.txt,67,call,0.658118,We therefore call on all our neighbouring coun...,With respect to the principles of non-interfer...
6,AFG,2016,AFG_71_2016.txt,95,call,0.658118,We call respectfully and amicably on all our f...,It is in that context that we have presented o...
7,AFG,2016,AFG_71_2016.txt,96,reiterate,0.765178,"In conclusion, I wish to reiterate, on behalf ...",We call respectfully and amicably on all our f...
8,AFG,2017,AFG_72_2017.txt,32,defended,0.607973,"We have borne adversity, deprivation and droug...",I am honoured to stand before the General Asse...
9,AFG,2017,AFG_72_2017.txt,40,call,0.658118,Only through political settlement can we achie...,The strategy consolidates all instruments of A...


In [13]:
CONTEXT_DIR = (
    PROJECT_ROOT
    / "data"
    / "contexts"
)

CONTEXT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [14]:
SA_CONTEXT_FILE = (
    CONTEXT_DIR
    / "development_self_assertion_contexts.csv"
)

sa_contexts.to_csv(
    SA_CONTEXT_FILE,
    index=False
)

print("Saved:")
print(SA_CONTEXT_FILE)

Saved:
../data/contexts/development_self_assertion_contexts.csv


In [1]:
from pathlib import Path
import pandas as pd

# Identify the project root
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Frozen KGL file
KGL_PATH = (
    PROJECT_ROOT
    / "data"
    / "purification"
    / "self_assertion_kgl_FINAL.csv"
)

print("Project root:", PROJECT_ROOT)
print("KGL exists:", KGL_PATH.is_file())

Project root: /Users/catherinexie/Documents/kgl-mockproject
KGL exists: True


In [2]:
assert KGL_PATH.is_file(), f"KGL file not found: {KGL_PATH}"

kgl = pd.read_csv(KGL_PATH)

print("Number of rows:", len(kgl))
print("Columns:", kgl.columns.tolist())

display(kgl.head())

assert len(kgl) == 62, (
    f"Expected 62 KGL rows, found {len(kgl)}"
)

Number of rows: 62
Columns: ['term', 'is_seed', 'sources', 'weight', 'token_count', 'internal_correlation_recomputed']


,term,is_seed,sources,weight,token_count,internal_correlation_recomputed
0,advocate,True,concept; seed,1.000000,306,0.009243
1,affirm,False,concept; synonym:reaffirm,0.823465,134,0.256554
2,appeal,False,concept,0.612561,381,0.084987
3,assert,True,concept; seed,1.000000,37,0.230127
4,asserted,False,concept,0.620962,13,0.274662


In [3]:
# Inspect available data files without loading them

data_dir = PROJECT_ROOT / "data"

files = sorted(
    p.relative_to(PROJECT_ROOT)
    for p in data_dir.rglob("*")
    if p.is_file()
    and p.suffix.lower() in {
        ".csv", ".parquet", ".json", ".jsonl", ".txt"
    }
)

for file in files:
    print(file)

data/contexts/development_self_assertion_contexts.csv
data/expansion/self_assertion_concept_expansion_primary.csv
data/expansion/self_assertion_expanded_raw_primary.csv
data/expansion/self_assertion_synonym_expansion_primary.csv
data/preprocessed/development_preprocessed.csv
data/preprocessed/development_sentences.csv
data/preprocessed/development_sentences_clean.csv
data/preprocessed/focal_preprocessed.csv
data/preprocessed/focal_sentences.csv
data/preprocessed/focal_sentences_clean.csv
data/processed/development_2016_2025.csv
data/processed/focal_2016_2025.csv
data/purification/self_assertion_frequency_purified.csv
data/purification/self_assertion_kgl_FINAL.csv
data/purification/self_assertion_kgl_candidate_final.csv
data/purification/self_assertion_semantic69_recomputed_correlations.csv
data/purification/self_assertion_semantic_term_review.csv
data/purification/self_assertion_semantic_term_review_FINAL.csv
data/seed_audit/Self Assertion Audit Manual.csv
data/seed_audit/Self Assertio

In [4]:
FOCAL_PATH = (
    PROJECT_ROOT
    / "data"
    / "preprocessed"
    / "focal_sentences_clean.csv"
)

assert FOCAL_PATH.is_file(), f"File not found: {FOCAL_PATH}"

focal = pd.read_csv(FOCAL_PATH)

print("Dataset shape:", focal.shape)
print("\nColumns:")
print(focal.columns.tolist())

print("\nFirst 5 rows:")
display(focal.head())

Dataset shape: (18882, 9)

Columns:
['corpus', 'iso3', 'session', 'year', 'filename', 'sentence_id', 'sentence_text', 'sentence_text_lower', 'procedural_intro']

First 5 rows:


,corpus,iso3,session,year,filename,sentence_id,sentence_text,sentence_text_lower,procedural_intro
0,focal,CHL,71,2016,CHL_71_2016.txt,0,We are living in times that show us the necess...,we are living in times that show us the necess...,False
1,focal,CHL,71,2016,CHL_71_2016.txt,1,Our world is in the midst of a crisis that ext...,our world is in the midst of a crisis that ext...,False
2,focal,CHL,71,2016,CHL_71_2016.txt,2,"At the same time, economic growth has given wa...","at the same time, economic growth has given wa...",False
3,focal,CHL,71,2016,CHL_71_2016.txt,3,It would appear that the far-reaching dynamics...,it would appear that the far-reaching dynamics...,False
4,focal,CHL,71,2016,CHL_71_2016.txt,4,But citizens today are more aware and empowered.,but citizens today are more aware and empowered.,False


In [5]:
sentence_files = [
    "focal_sentences.csv",
    "focal_sentences_clean.csv"
]

for filename in sentence_files:
    path = PROJECT_ROOT / "data" / "preprocessed" / filename

    df = pd.read_csv(path)

    print(f"\n{'=' * 60}")
    print(filename)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    display(df.head(3))


focal_sentences.csv
Shape: (18903, 8)
Columns: ['corpus', 'iso3', 'session', 'year', 'filename', 'sentence_id', 'sentence_text', 'sentence_text_lower']


,corpus,iso3,session,year,filename,sentence_id,sentence_text,sentence_text_lower
0,focal,CHL,71,2016,CHL_71_2016.txt,0,We are living in times that show us the necess...,we are living in times that show us the necess...
1,focal,CHL,71,2016,CHL_71_2016.txt,1,Our world is in the midst of a crisis that ext...,our world is in the midst of a crisis that ext...
2,focal,CHL,71,2016,CHL_71_2016.txt,2,"At the same time, economic growth has given wa...","at the same time, economic growth has given wa..."



focal_sentences_clean.csv
Shape: (18882, 9)
Columns: ['corpus', 'iso3', 'session', 'year', 'filename', 'sentence_id', 'sentence_text', 'sentence_text_lower', 'procedural_intro']


,corpus,iso3,session,year,filename,sentence_id,sentence_text,sentence_text_lower,procedural_intro
0,focal,CHL,71,2016,CHL_71_2016.txt,0,We are living in times that show us the necess...,we are living in times that show us the necess...,False
1,focal,CHL,71,2016,CHL_71_2016.txt,1,Our world is in the midst of a crisis that ext...,our world is in the midst of a crisis that ext...,False
2,focal,CHL,71,2016,CHL_71_2016.txt,2,"At the same time, economic growth has given wa...","at the same time, economic growth has given wa...",False


In [6]:
# Expected focal sample
FOCAL_COUNTRIES = {
    "USA", "GBR", "CHN", "JPN", "SAU", "JOR",
    "CHL", "COL", "IND", "PAK", "GHA", "KEN",
    "IDN", "THA", "KAZ", "TJK", "POL", "ROU"
}

FOCAL_YEARS = set(range(2016, 2026))

# Identify unique speeches
speeches = focal[
    ["iso3", "year", "filename"]
].drop_duplicates()

# Check expected coverage
assert set(focal["iso3"]) == FOCAL_COUNTRIES, (
    "Country coverage does not match the focal sample."
)

assert set(focal["year"]) == FOCAL_YEARS, (
    "Year coverage does not match 2016–2025."
)

assert len(speeches) == 180, (
    f"Expected 180 speeches, found {len(speeches)}."
)

# Each country-year should have exactly one speech
country_year_counts = (
    speeches.groupby(["iso3", "year"])
    .size()
)

assert (country_year_counts == 1).all(), (
    "Some country-year combinations have duplicate speeches."
)

# Check sentence identifiers
assert not focal.duplicated(
    ["filename", "sentence_id"]
).any(), "Duplicate sentence identifiers found."

assert focal["sentence_text_lower"].notna().all(), (
    "Missing sentence text found."
)

print("All focal corpus checks passed.")
print("Countries:", speeches["iso3"].nunique())
print("Years:", speeches["year"].nunique())
print("Speeches:", len(speeches))
print("Cleaned sentences:", len(focal))

All focal corpus checks passed.
Countries: 18
Years: 10
Speeches: 180
Cleaned sentences: 18882


In [7]:
# Verify required columns
assert {"term", "weight"}.issubset(kgl.columns), (
    "Missing required KGL columns."
)

# Check missing values
assert kgl["term"].notna().all(), "Missing KGL terms."
assert kgl["weight"].notna().all(), "Missing KGL weights."

# Check duplicated terms
terms = kgl["term"].astype(str).str.strip().str.lower()

assert not terms.duplicated().any(), (
    "Duplicate KGL terms found."
)

assert (terms != "").all(), "Empty KGL terms found."

# Examine term structure
print("Total terms:", len(terms))
print("Single-word terms:", (terms.str.split().str.len() == 1).sum())
print("Multiword terms:", (terms.str.split().str.len() > 1).sum())

print("\nMultiword terms:")
print(terms[terms.str.split().str.len() > 1].tolist())

print("\nWeight range:")
print(kgl["weight"].agg(["min", "max"]))

print("\nFrozen KGL validation passed.")

Total terms: 62
Single-word terms: 62
Multiword terms: 0

Multiword terms:
[]

Weight range:
min    0.607973
max    1.000000
Name: weight, dtype: float64

Frozen KGL validation passed.


In [8]:
import re

# Compile one regex for all 62 terms
terms = kgl["term"].str.lower().str.strip().tolist()

pattern = re.compile(
    r"\b(?:" + "|".join(re.escape(t) for t in terms) + r")\b",
    flags=re.IGNORECASE
)

# Find all KGL term occurrences in each sentence
focal["matched_terms"] = focal["sentence_text_lower"].apply(
    lambda text: [m.group(0).lower() for m in pattern.finditer(text)]
)

# Extract sentences containing at least one match
contexts = focal.loc[
    focal["matched_terms"].str.len() > 0,
    [
        "iso3", "year", "filename", "sentence_id",
        "sentence_text", "matched_terms"
    ]
].copy()

# Number of matched term occurrences in each sentence
contexts["n_matches"] = contexts["matched_terms"].str.len()

print("Total focal sentences:", len(focal))
print("Self-Assertion sentences:", len(contexts))
print("Total KGL term occurrences:", contexts["n_matches"].sum())

display(contexts.head(10))

Total focal sentences: 18882
Self-Assertion sentences: 1230
Total KGL term occurrences: 1309


,iso3,year,filename,sentence_id,sentence_text,matched_terms,n_matches
22,CHL,2016,CHL_71_2016.txt,22,I insist that we must act now.,[insist],1
28,CHL,2016,CHL_71_2016.txt,28,I am also speaking from experience of having e...,[requested],1
38,CHL,2016,CHL_71_2016.txt,38,Our nation’s principles are immutable and incl...,[call],1
50,CHL,2016,CHL_71_2016.txt,50,Chile has therefore decided to take action by ...,[decided],1
53,CHL,2016,CHL_71_2016.txt,53,That is why I raise my voice again in favour o...,[favour],1
54,CHL,2016,CHL_71_2016.txt,54,I raise my voice in favour of an effective man...,[favour],1
63,CHL,2016,CHL_71_2016.txt,63,Let us make the United Nations the Organizatio...,[call],1
83,CHL,2017,CHL_72_2017.txt,19,The enormous strength of the 2030 Agenda for S...,"[propose, propose]",2
98,CHL,2017,CHL_72_2017.txt,34,We will thus be the first country in America t...,[call],1
113,CHL,2017,CHL_72_2017.txt,49,But we understand that strengthening our democ...,[demand],1


In [9]:
import json

OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "contexts"
    / "focal_self_assertion_contexts.csv"
)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Save matched terms as JSON strings for reliable reloading
contexts_to_save = contexts.copy()
contexts_to_save["matched_terms"] = contexts_to_save["matched_terms"].apply(
    json.dumps
)

contexts_to_save.to_csv(OUTPUT_PATH, index=False)

print(f"Saved {len(contexts_to_save)} passages to:")
print(OUTPUT_PATH)

Saved 1230 passages to:
/Users/catherinexie/Documents/kgl-mockproject/data/contexts/focal_self_assertion_contexts.csv


In [10]:
speech_counts = (
    focal.groupby(["iso3", "year", "filename"])
    .agg(
        n_sentences=("sentence_id", "size"),
        n_matches=("matched_terms", lambda x: sum(len(t) for t in x))
    )
    .reset_index()
)

print("Speeches with zero matches:",
      (speech_counts["n_matches"] == 0).sum())

print("\nMatches per speech:")
print(speech_counts["n_matches"].describe())

print("\nSpeeches with fewer than 3 matches:",
      (speech_counts["n_matches"] < 3).sum())

Speeches with zero matches: 3

Matches per speech:
count    180.000000
mean       7.272222
std        5.274404
min        0.000000
25%        3.000000
50%        7.000000
75%       10.000000
max       30.000000
Name: n_matches, dtype: float64

Speeches with fewer than 3 matches: 31
